# PT-W4-D6 实验 · 组装 Semantic Model v0.1 + 验证设计

**概念问题**：六层模型各自都对，拼在一起就对吗？组装不是堆叠，是**接线**——层间每一根引用都不悬空（六条装配规则 W1-W6）；
验证不是"跑一遍看顺不顺"，是设计**能让模型失败**的测试（三组对照夹具 + L1→L5 轨迹判定梯）。

本 notebook 做五件事：
1. 把 D1-D5 六层成果写成可运行的精简结构（Ontology / Lifecycle / Rule / Capability+Policy / Agent）
2. 跑**六条装配规则审计器**——含一次故意植入的悬空引用拦截演示
3. 构造三组对照夹具：A101 反例 / A102 正例 / A103 边界例（快照陷阱杀手）
4. 跑 L1→L5 轨迹推理引擎——验证对象是**轨迹**（每一跳带 evidence ID），不是答案
5. 判分表输出 + 层间引用图 + 判分热力图


In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

## 1. 六层语义模型（精简版，D1-D5 成果汇编）

今天不再是"某一层"，而是把五天的产出放进同一个命名空间——**这一天才是"组装"的开始**：

- **Ontology（D1）**：Entity 带业务定义 + 身份路径（A101 的身份是 `Project→Building→Floor→Unit`，不是主键）
- **Lifecycle（D2）**：迁移声明 `from→to / guard / emits / effects`，effects 必须落在 effect-registry 冻结的 5 类之内
- **Rule（D3）**：四物种 Rule Card，谓词写成 `Entity.属性 OP 值` 的结构化形式（供 W1 审计解析）

In [ ]:
from dataclasses import dataclass, field

# ── D1 Ontology 层 ──────────────────────────────────────────
@dataclass
class EntityDef:
    name: str
    definition: str          # 业务定义（一句话"这是什么"）
    identity: tuple          # 业务身份层级
    attributes: list         # 属性名（谓词可引用的符号）

ENTITIES = {
    "Space": EntityDef("Space", "商场中可被独立出租、计价与运营的最小空间单元",
                       ("Project", "Building", "Floor", "Unit"), ["asset_status", "restriction_flag"]),
    "Lease": EntityDef("Lease", "某段时间内占住铺位的合同事实，铺位可租性由它决定",
                       ("Project", "LeaseNo"), ["state", "inspection_status", "settlement_status"]),
    "Reservation": EntityDef("Reservation", "意向客户对铺位的短期锁定",
                             ("Project", "ResNo"), ["state", "expiry_date"]),
    "InspectionTask": EntityDef("InspectionTask", "退场验收任务，终止守卫的检查载体",
                                ("Project", "TaskNo"), ["status"]),
}
RELATIONSHIPS = {  # 语义动词命名（D1 升级）
    ("Lease", "occupies", "Space"), ("Reservation", "holds", "Space"),
    ("InspectionTask", "verifies", "Lease"),
}
# ── effect-registry v1.0 冻结的 5 类（真实文件：effect-registry.yaml）──
FROZEN_EFFECTS = {"occupancy", "financial", "state-transition", "lead-conversion", "maintenance"}

# ── D2 Lifecycle 层：统一迁移声明 ────────────────────────────
@dataclass
class Transition:
    tid: str; entity: str; frm: str; to: str
    guard_rules: list; emits: str; effects: list

TRANSITIONS = [
    Transition("T1", "Lease", "Active", "Terminating", [], "TerminationSubmitted", ["state-transition"]),
    Transition("T2", "Lease", "Terminating", "Terminated", ["CRE-R-003"], "ContractTerminated",
               ["occupancy", "financial"]),                       # 释放铺位 + 停止计费
    Transition("T3", "Reservation", "Reserved", "Released", ["CRE-R-008"], "ReservationExpired",
               ["occupancy"]),                                    # 预定到期自动释放
    Transition("T4", "Lease", "Signed", "Active", ["CRE-R-019"], "MoveInCompleted",
               ["state-transition"]),                              # 进场迁移（守卫=进场守卫）
]

# ── D3 Rule 层：四物种 Rule Card ────────────────────────────
@dataclass
class RuleCard:
    rid: str; species: str; statement: str
    predicates: list           # "Entity.attr OP value"，供 W1 符号解析
    mount: str = None          # Guard 的挂载点（迁移 ID），供 W2
    variant_of: str = None     # Variant 的 base 规则，供 W6
    evidence: str = ""

RULES = {
    "CRE-R-001": RuleCard("CRE-R-001", "Invariant", "一个 RU 同一时间只有一个 Active Occupancy",
                          ["Lease.state == Active"], evidence="Domain Model Lease/Occupancy 不变量 1"),
    "CRE-R-002": RuleCard("CRE-R-002", "Derivation",
                          "AvailableForLeasing = Asset.Active ∧ no Occupancy ∧ no Restriction",
                          ["Space.asset_status == Active", "Space.restriction_flag == absent",
                           "Lease.state notin Active,Terminating", "Reservation.state notin Reserved"],
                          evidence="Domain Model 领域规则（D-001 Amendment A：不落库，永远重算）"),
    "CRE-R-003": RuleCard("CRE-R-003", "Guard", "终止迁移前：Inspection 完成 ∧ 清算完成",
                          ["Lease.inspection_status == completed", "Lease.settlement_status == completed"],
                          mount="T2", evidence="CRE-LEA-011 + 03-租赁管理 §8.3"),
    "CRE-R-008": RuleCard("CRE-R-008", "Guard", "预定到期自动释放；生效期间他人不可选",
                          ["Reservation.expiry_date < today"], mount="T3", evidence="CRE-LEA-008"),
    "CRE-R-019": RuleCard("CRE-R-019", "Guard", "进场守卫（base）：合同生效方可进场",
                          ["Lease.state == Active"], mount="T4", evidence="CRE-LEA-011 base"),
    "CRE-R-020": RuleCard("CRE-R-020", "Variant", "进场守卫：万达=首期欠缴不许进场 / 中旅=未挂表不许开业",
                          ["Lease.state == Active"], variant_of="CRE-R-019", evidence="CRE-LEA-011 变体"),
}
print(f"D1: {len(ENTITIES)} Entity / {len(RELATIONSHIPS)} 关系 | D2: {len(TRANSITIONS)} 迁移 | D3: {len(RULES)} Rule Card")

In [ ]:
# ── D4 Capability + Policy 层 ────────────────────────────────
@dataclass
class CapabilityCard:
    cid: str; verb: str; obj: str; context: str
    status: str          # MI 追溯矩阵词表：accepted / spec-defined / excluded
    delegation: str      # Policy③ 委托边界：full / conditional / none
    human_review_gate: bool  # Policy② AI 执行姿态

CAPABILITIES = {
    "ops.inspection.create": CapabilityCard("ops.inspection.create", "创建", "退场验收任务", "05-运营管理",
                                            "accepted", "conditional", True),
    "lease.termination.submit": CapabilityCard("lease.termination.submit", "提交", "终止申请", "02-合同管理",
                                               "accepted", "conditional", True),
    "membership.migrate": CapabilityCard("membership.migrate", "迁移", "会员卡/积分", "06-商户管理",
                                         "excluded", "none", True),   # MI 矩阵 excluded：硬闸门
}
@dataclass
class Skill:
    sid: str; capability_dependencies: list; status: str   # 状态诚实：全部（候选）

SKILLS = {"ops.inspection.manage": Skill("ops.inspection.manage", ["ops.inspection.create"], "候选")}

# ── D5 Agent Mapping 层 ─────────────────────────────────────
@dataclass
class AgentCard:
    name: str; anchor: str            # 岗位锚点（业务岗位轴，非 Skill 轴）
    contexts: dict                    # 认知边界：Context × 读写姿态
    skills: list; status: str

CONTEXTS17 = {"01-招商管理","02-合同管理","03-租赁管理","04-财务管理","05-运营管理","06-商户管理",
              "07-工程管理","08-物业管理","09-安全管理","10-停车场","11-会员运营","12-营销",
              "13-数据报表","14-系统配置","15-工单","16-审批","17-集成"}  # 17 Bounded Context 精简示意

AGENTS = {
    "运营数字员工": AgentCard("运营数字员工", "运营",
        {"03-租赁管理": "RW", "02-合同管理": "RO", "05-运营管理": "RW", "06-商户管理": "RO"},
        ["ops.inspection.manage"], "规划"),
    "招商数字员工": AgentCard("招商数字员工", "招商",
        {"01-招商管理": "RW", "03-租赁管理": "RO", "02-合同管理": "RO"}, [], "规划"),
}
print(f"D4: {len(CAPABILITIES)} Capability / {len(SKILLS)} Skill | D5: {len(AGENTS)} Agent Card")

## 2. 六条装配规则审计器（组装 = 符号解析）

组装的本质是**层间引用完整性**——每一根箭头都可检查、可悬空、可拦截：

| 规则 | 检查 | 本实现的判据 |
|---|---|---|
| W1 | Rule 谓词符号可解析 | `Entity.attr` 中 Entity ∈ ENTITIES 且 attr ∈ attributes |
| W2 | Guard 挂载点存在 | mount ∈ 迁移 ID 集合 |
| W3 | effects 类型封闭 | ⊆ effect-registry 冻结 5 类 |
| W4 | Skill 依赖可解析且非 excluded | 依赖 ⊆ Capability Map，status ≠ excluded |
| W5 | 认知边界合法 | Agent contexts ⊆ 17 Context；技能依赖能力落在边界内 |
| W6 | Variant 锚定 base | variant_of 指向存在的非 Variant 规则 |

先跑一遍应全绿；然后**故意植入一条悬空规则**（谓词引用不存在的属性 + 挂在不存在的迁移上），看审计器拦截。

In [ ]:
import re

def audit_w1(rules, entities):
    issues, n = [], 0
    for rid, rc in rules.items():
        for p in rc.predicates:
            m = re.match(r"(\w+)\.(\w+)", p)
            if not m: issues.append(f"W1: {rid} 谓词无法解析: {p}"); continue
            ent, attr = m.groups(); n += 1
            if ent not in entities: issues.append(f"W1: {rid} 引用未定义 Entity: {ent}")
            elif attr not in entities[ent].attributes: issues.append(f"W1: {rid} 引用未定义属性: {ent}.{attr}")
    return issues, n

def audit_w2(rules, transitions):
    tids = {t.tid for t in transitions}; issues, n = [], 0
    for rid, rc in rules.items():
        if rc.species == "Guard":
            n += 1
            if rc.mount not in tids: issues.append(f"W2: Guard {rid} 挂载点悬空: {rc.mount}")
    return issues, n

def audit_w3(transitions, frozen):
    issues, n = [], 0
    for t in transitions:
        for e in t.effects:
            n += 1
            if e not in frozen: issues.append(f"W3: {t.tid} effect 越界: {e}")
    return issues, n

def audit_w4(skills, caps):
    issues, n = [], 0
    for sid, sk in skills.items():
        for cid in sk.capability_dependencies:
            n += 1
            if cid not in caps: issues.append(f"W4: Skill {sid} 依赖悬空: {cid}")
            elif caps[cid].status == "excluded": issues.append(f"W4: Skill {sid} 依赖 excluded 能力: {cid}")
    return issues, n

def audit_w5(agents, contexts17, skills, caps):
    issues, n = [], 0
    for name, ag in agents.items():
        for ctx in ag.contexts:
            n += 1
            if ctx not in contexts17: issues.append(f"W5: {name} 认知边界越界: {ctx}")
        for s in ag.skills:
            for cid in skills[s].capability_dependencies:
                n += 1
                if caps[cid].context not in ag.contexts:
                    issues.append(f"W5: {name} 依赖边界外能力: {cid}({caps[cid].context})")
    return issues, n

def audit_w6(rules):
    issues, n = [], 0
    for rid, rc in rules.items():
        if rc.species == "Variant":
            n += 1
            base = rules.get(rc.variant_of)
            if base is None: issues.append(f"W6: Variant {rid} 无 base: {rc.variant_of}")
            elif base.species == "Variant": issues.append(f"W6: Variant {rid} 的 base 也是 Variant")
    return issues, n

def full_audit(rules=RULES, transitions=TRANSITIONS, entities=ENTITIES, skills=SKILLS, caps=CAPABILITIES, agents=AGENTS):
    results = [audit_w1(rules, entities), audit_w2(rules, transitions), audit_w3(transitions, FROZEN_EFFECTS),
               audit_w4(skills, caps), audit_w5(agents, CONTEXTS17, skills, caps), audit_w6(rules)]
    all_issues, ref_counts = [], {}
    for (issues, n), wname in zip(results, ["W1谓词符号", "W2挂载点", "W3效果类型", "W4技能依赖", "W5认知边界", "W6变体锚定"]):
        ref_counts[wname] = n; all_issues += issues
    return all_issues, ref_counts

issues, refs = full_audit()
print("── 第一次审计（模型本体）──")
print("层间引用计数:", refs)
print("结果:", "✅ 全绿，无悬空引用" if not issues else "❌ " + "; ".join(issues))

# 故意植入悬空引用：谓词引用不存在属性 reservation_status，挂在不存在的迁移 T99
print("\n── 植入悬空规则 CRE-R-099 后再审 ──")
bad_rules = dict(RULES)
bad_rules["CRE-R-099"] = RuleCard("CRE-R-099", "Guard", "预定过期后 7 日内不可再预定（杜撰）",
                                  ["Space.reservation_status == expired"], mount="T99", evidence="无中生有")
issues2, _ = full_audit(rules=bad_rules)
for i in issues2: print("拦截:", i)
print("结论: 悬空引用被 W1+W2 双重拦截 —— 组装规则就是语义模型的符号解析")

## 3. 三组对照夹具：让模型"能够失败"

单夹具、单方向、无对照的验证等于没验。三组夹具各杀一种退化：

- **A101 反例**：退租流程卡住（Terminating + Inspection 未完成）→ 预期**不可租**，考验失败分支解释
- **A102 正例**：干净铺位 → 预期**可租**，防止"永远说不租"的退化模型
- **A103 边界例**：预定已过期（到期日 8/20 < 今天 8/22）→ 预期**可租**，且结论必须来自 **CRE-R-008 释放事件**而非字段快照——专杀"快照式实现"（D-001 Amendment A）

In [ ]:
from datetime import date
TODAY = date(2026, 8, 22)   # 固定"今天"，保证夹具可复现

def fixture(space_id, path, asset_status, restriction, occupancy=None, reservation=None, snapshot_available=None):
    return {"space_id": space_id, "path": path, "asset_status": asset_status,
            "restriction_flag": restriction, "active_occupancy": occupancy,
            "reservation": reservation, "snapshot_available": snapshot_available}  # snapshot = 落库的过期快照

WORLDS = {
    "A101 反例": fixture("A101", ("万达广场-XX店", "1号楼", "F1", "A101"), "Active", "absent",
        occupancy={"lease": "L-2024-088", "state": "Terminating",
                   "inspection_status": "pending", "settlement_status": "completed"},
        snapshot_available=False),
    "A102 正例": fixture("A102", ("万达广场-XX店", "1号楼", "F1", "A102"), "Active", "absent",
        occupancy=None, reservation=None, snapshot_available=True),
    "A103 边界例": fixture("A103", ("万达广场-XX店", "2号楼", "B1", "A103"), "Active", "absent",
        occupancy=None, reservation={"res": "R-0917", "state": "Reserved", "expiry": date(2026, 8, 20)},
        snapshot_available=False),   # ← 快照还停留在"被预定"，没人刷新
}
for k, w in WORLDS.items():
    occ = w["active_occupancy"]["lease"] + "/" + w["active_occupancy"]["state"] if w["active_occupancy"] else "无"
    res = w["reservation"]["res"] + " 到期 " + str(w["reservation"]["expiry"]) if w["reservation"] else "无"
    print(f"{k}: {w['space_id']} | occupancy: {occ} | reservation: {res} | 落库快照 available={w['snapshot_available']}")

print("\n── 快照陷阱演示（A103）──")
print(f"落库快照说: available={WORLDS['A103 边界例']['snapshot_available']}（预定期间写入，之后无人刷新）")
print(f"事实是: 预定到期日 2026-08-20 < 今天 {TODAY} —— ReservationExpired 事件已发生，铺位已释放")
print("=> 信任快照会答错；Derivation 永远从事实重算（CRE-R-002），这就是 D-001 Amendment A 的裁决")

## 4. L1→L5 轨迹推理引擎：验证对象是轨迹，不是答案

每一级返回一个轨迹步：`(级别, 结论, evidence ID, 判定)`。**答案可以是语感撞对的，轨迹撞不了**——这是 D7 验证报告的主体。

In [ ]:
def L1(w):  # 语义理解：身份路径解析
    return ("L1", "身份路径 " + " → ".join(w["path"]), "D1 Identity 层级身份", "pass")

def L2(w):  # 业务链推理：关系遍历 + 状态机节点定位
    if w["active_occupancy"]:
        occ = w["active_occupancy"]
        return ("L2", f"occupies ← Lease {occ['lease']}，状态机节点 = {occ['state']}（还差迁移 T2 才释放铺位）",
                "T2: Terminating→Terminated", "pass")
    if w["reservation"]:
        if w["reservation"]["expiry"] < TODAY:
            return ("L2", f"holds ← Reservation {w['reservation']['res']}，节点 = Released（ReservationExpired 事件已触发 T3）",
                    "T3: Reserved→Released + CRE-R-008", "pass")
        return ("L2", f"holds ← Reservation {w['reservation']['res']}，节点 = Reserved（生效中）", "T3", "pass")
    return ("L2", "无 occupancy / 无 reservation，Space 处于 Vacant", "D2 状态机", "pass")

def L3(w):  # 规则判断：Derivation 重算 + 失败分支定位
    if w["asset_status"] != "Active" or w["restriction_flag"] != "absent":
        return ("L3", "不可租（资产状态/限制）", "CRE-R-002", "pass")
    if w["active_occupancy"]:
        occ = w["active_occupancy"]
        if occ["state"] == "Terminating":
            failed = [p for p, v in [("Inspection", occ["inspection_status"]), ("清算", occ["settlement_status"])] if v != "completed"]
            return ("L3", f"推导 False：存在未完成退租流程（守卫未满足：{'、'.join(failed)}）",
                    "CRE-R-002 失败分支 → CRE-R-003@T2", "pass")
        return ("L3", "推导 False：存在 Active Occupancy", "CRE-R-002 + CRE-R-001", "pass")
    if w["reservation"] and w["reservation"]["expiry"] >= TODAY:
        return ("L3", "推导 False：预定生效中，他人不可选", "CRE-R-002 + CRE-R-008", "pass")
    return ("L3", "推导 True：Active ∧ 无 Occupancy ∧ 无 Restriction（含已释放预定）", "CRE-R-002", "pass")

def L4(w):  # Policy 判断：审批路径裁决
    if w["active_occupancy"] and w["active_occupancy"]["state"] == "Terminating":
        return ("L4", "创建验收任务免审批（终止申请 K2 已过；巡检创建走 ② 层 conditional 授权）",
                "Policy② CRE-CON-024 关联条款", "pass")
    return ("L4", "不适用（无需推进动作）", "—", "n/a")

def L5(w):  # 动作建议：能力 + 姿态 + 边界三查
    if w["active_occupancy"] and w["active_occupancy"]["state"] == "Terminating":
        cap = CAPABILITIES["ops.inspection.create"]
        agent = AGENTS["运营数字员工"]
        in_boundary = cap.context in agent.contexts          # W5 的运行时版本
        posture_ok = cap.delegation == "conditional" and cap.human_review_gate
        if in_boundary and posture_ok:
            return ("L5", f"建议 {cap.cid}（{cap.verb}{cap.obj}）：conditional_write + human_review_gate，"
                          f"由{agent.name}发起（{cap.context} 边界内）", f"{cap.cid} + AgentCard[运营]", "pass")
    return ("L5", "不适用（无可建议动作）", "—", "n/a")

def run_trace(name):
    w = WORLDS[name]
    print(f"\n━━━ {name}：「{w['space_id']} 铺位为什么不能出租 / 能不能租？」━━━")
    for fn in (L1, L2, L3, L4, L5):
        lv, desc, ev, verdict = fn(w)
        print(f"  {lv} [{verdict:>4}] {desc}\n        evidence: {ev}")

run_trace("A101 反例")

In [ ]:
run_trace("A102 正例")
run_trace("A103 边界例")

## 5. 判分表（rubric）：预期 × 轨迹的逐格比对

- A101 是全级别考场（L1-L5 全部适用）
- A102/A103 主要杀 **L3 退化**（"永远说不租" / "信任快照"），L4/L5 不适用记 n/a
- 判分对象是轨迹步的 verdict + evidence，不是最终答案文本

In [ ]:
EXPECTED = {  # 验证设计预先写死的预期（含"如果模型错了会是什么样"）
    "A101 反例":  {"L1": "pass", "L2": "pass", "L3": "pass", "L4": "pass", "L5": "pass"},
    "A102 正例":  {"L1": "pass", "L2": "pass", "L3": "pass", "L4": "n/a",  "L5": "n/a"},
    "A103 边界例": {"L1": "pass", "L2": "pass", "L3": "pass", "L4": "n/a",  "L5": "n/a"},
}
WEIGHT = {"L1": 10, "L2": 20, "L3": 25, "L4": 15, "L5": 30}

def verdicts(name):
    w = WORLDS[name]
    return {fn.__name__.upper(): fn(w)[3] for fn in (L1, L2, L3, L4, L5)}

rows, total,满分 = [], 0, 0
print(f"{'夹具':<10}{'L1':>6}{'L2':>6}{'L3':>6}{'L4':>6}{'L5':>6}   判分")
for name, exp in EXPECTED.items():
    got = verdicts(name)
    ok = all(got[k] == v for k, v in exp.items())
    applicable = {k: WEIGHT[k] for k, v in exp.items() if v == "pass"}
    score = sum(applicable.values()) if ok else 0
    total += score; 满分 += sum(applicable.values())
    rows.append([name] + [got[f"L{i}"] for i in range(1, 6)] + [score])
    print(f"{name:<12}" + "".join(f"{got[f'L{i}']:>6}" for i in range(1, 6)) + f"   {'✅ 通过' if ok else '❌ 挂'} ({score} 分)")
print(f"\n总分 {total}/{满分}（L4/L5 权重计入适用夹具）—— 必达项 L1-L3 全绿，L5 超额达成")

In [ ]:
import numpy as np
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))

# 左图：层间引用计数（第一次审计结果）—— 六条装配规则各解析了多少根引用线
names = list(refs.keys()); vals = list(refs.values())
colors = ["#4C78A8"] * len(names)
ax1.bar(names, vals, color=colors)
for i, v in enumerate(vals): ax1.text(i, v + 0.15, str(v), ha="center", fontsize=10)
ax1.set_title("组装 = 符号解析：六条装配规则的层间引用计数", fontsize=12)
ax1.set_ylabel("解析的引用数（全部命中定义）")
ax1.tick_params(axis="x", rotation=20)

# 右图：判分热力图（夹具 × 判定梯）
data = np.array([[{"pass": 2.0, "n/a": 1.0, "fail": 0.0}[rows[r][c]] for c in range(1, 6)] for r in range(3)])
from matplotlib.colors import ListedColormap
cmap = ListedColormap(["#E45756", "#BAB0AC", "#54A24B"])
ax2.imshow(data, cmap=cmap, vmin=0, vmax=2, aspect="auto")
ax2.set_xticks(range(5)); ax2.set_xticklabels([f"L{i+1}" for i in range(5)])
ax2.set_yticks(range(3)); ax2.set_yticklabels([r[0] for r in rows])
label = {2.0: "通过", 1.0: "不适用", 0.0: "挂"}
for r in range(3):
    for c in range(5):
        ax2.text(c, r, label[data[r, c]], ha="center", va="center",
                 color="white" if data[r, c] == 2 else "#333", fontsize=11)
ax2.set_title("验证设计判分矩阵：三组对照夹具 × L1→L5 判定梯", fontsize=12)
plt.tight_layout(); plt.savefig("/root/learning-notebooks/第12周/d6_assembly_validation.png", dpi=120)
plt.show()
print("已保存: d6_assembly_validation.png")

## 6. 收束：组装完成 ≠ 交付完成

- **六条装配规则全绿** = 语义模型通过了"符号解析"——推理链任何一跳不会断链，悬空引用被双重拦截（W1 符号 + W2 挂载点）
- **三组对照夹具** = 测试套件：正例防退化、反例考验失败分支、边界例专杀快照式实现（A103 快照说不可租、事实已释放）
- **判定梯判的是轨迹**：每一跳带 evidence ID（Rule ID / 迁移 ID / capability ID / Agent Card），答案正确但轨迹无据 = 挂

**架构师判断**：模型组装完才是编译的开始——装配规则是编译器前端，夹具是测试套件，判定梯是 CI，trace 是被测 IR。
"能被验证的语义模型"和"能被测试的代码"完全同构：没有夹具的模型等于没有测试的代码。

明天 D7：用这套验证设计跑 Digital Employee Validation，出验证报告。